# Modélisation de données avec des arbres de décision

Ce notebook sert à découvrir comment un Random Forest peut prédire les comptages de vélos à Paris.

Après avoir travaillé avec les modèles linéaires, l’idée ici est de voir ce qu’un modèle non linéaire peut apporter.

Le modèle utilisé est :

**Random Forest Regressor** : un ensemble d’arbres de décision qui apprennent chacun une version légèrement différente des données, puis combinent leurs prédictions. Cette technique :

  * repère des relations non linéaires,
  * limite le surapprentissage,
  * fonctionne bien même si les données sont variées.

Comme pour les modèles précédents, les performances sont mesurées avec la **MAE**, la **RMSE** et le **R²**.

Ce notebook prépare le terrain pour les modèles encore plus puissants comme le gradient boosting, que l’on testera ensuite pour aller plus loin dans la modélisation.

## Import des librairies et des données

In [1]:
import time
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [2]:
df = pd.read_csv('../data/processed/df_processed.csv', sep=',')

In [3]:
df.head()

,Nom du compteur,Nom du site de comptage,Comptage horaire,Date et heure de comptage,Lien vers photo du site de comptage,Direction,Latitude,Longitude,Température (°C),Précipitations (mm),...,Mois,Année,Heure,Jour de la semaine,Week-end,Vacances,lag_1h,lag_24h,lag_168h,roll_mean_3h
0,10 avenue de la Grande Armée 10 avenue de la G...,10 avenue de la Grande Armée,0,2025-01-07 11:00:00,https://filer.eco-counter-tools.com/file/26/1d...,Bike IN,48.8748,2.2924,4.3,1.8,...,1,2025,11,1,0,0,NaN,NaN,NaN,NaN
1,10 avenue de la Grande Armée 10 avenue de la G...,10 avenue de la Grande Armée,13,2025-01-07 12:00:00,https://filer.eco-counter-tools.com/file/26/1d...,Bike IN,48.8748,2.2924,5.4,0.0,...,1,2025,12,1,0,0,0.0,NaN,NaN,NaN
2,10 avenue de la Grande Armée 10 avenue de la G...,10 avenue de la Grande Armée,50,2025-01-07 13:00:00,https://filer.eco-counter-tools.com/file/26/1d...,Bike IN,48.8748,2.2924,6.3,0.0,...,1,2025,13,1,0,0,13.0,NaN,NaN,NaN
3,10 avenue de la Grande Armée 10 avenue de la G...,10 avenue de la Grande Armée,54,2025-01-07 14:00:00,https://filer.eco-counter-tools.com/file/26/1d...,Bike IN,48.8748,2.2924,6.8,0.5,...,1,2025,14,1,0,0,50.0,NaN,NaN,21.0
4,10 avenue de la Grande Armée 10 avenue de la G...,10 avenue de la Grande Armée,33,2025-01-07 15:00:00,https://filer.eco-counter-tools.com/file/26/1d...,Bike IN,48.8748,2.2924,6.9,0.0,...,1,2025,15,1,0,0,54.0,NaN,NaN,39.0


## Préparation des données avant entraînement

In [4]:
df.columns

Index(['Nom du compteur', 'Nom du site de comptage', 'Comptage horaire',
       'Date et heure de comptage', 'Lien vers photo du site de comptage',
       'Direction', 'Latitude', 'Longitude', 'Température (°C)',
       'Précipitations (mm)', 'Jour du mois', 'Mois', 'Année', 'Heure',
       'Jour de la semaine', 'Week-end', 'Vacances', 'lag_1h', 'lag_24h',
       'lag_168h', 'roll_mean_3h'],
      dtype='object')

In [5]:
feature_cols_num = [
    'Année', 'Mois', 'Jour du mois', 'Heure', 'Jour de la semaine',
    'Week-end', 'Vacances', 'lag_1h', 'lag_24h', 'lag_168h',
    'roll_mean_3h', 'Température (°C)', 'Précipitations (mm)',
]

feature_cols_cat = ['Nom du compteur', 'Direction']

for col in feature_cols_num:
    df[col] = pd.to_numeric(df[col], errors='coerce')
# Assure que toutes les colonnes numériques sont bien en format numérique

df_clean = df.dropna(subset=feature_cols_num + feature_cols_cat + ['Comptage horaire']).copy()
# Supprime les valeurs vides (en particulier celles des colonnes lag)

df['Date et heure de comptage'] = pd.to_datetime(df['Date et heure de comptage'], errors='coerce')
# Convertit 'Date et heure de comptage' en format datetime

df_clean = df_clean.sort_values('Date et heure de comptage')

split = int(len(df_clean) * 0.8)
train = df_clean.iloc[:split]
test = df_clean.iloc[split:]
# Sépare le dataset chronologiquement

X_train = train[feature_cols_num + feature_cols_cat]
y_train = train['Comptage horaire']
X_test = test[feature_cols_num + feature_cols_cat]
y_test = test['Comptage horaire']

# Création d'un pipeline : on convertit les variables catégorielles en vecteurs numériques.
preprocess = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', feature_cols_num),
        #('num', StandardScaler(), feature_cols_num), La standardisation des variables numériques n'est pas nécessaire pour Random Forest
        ('cat', OneHotEncoder(handle_unknown='ignore'), feature_cols_cat),
    ],
    remainder='drop',
)

## Entraînement de modèle (1) : Random Forest

In [6]:
rf_base = RandomForestRegressor(
    n_jobs=-1,
    random_state=42,
    verbose=0,
    max_depth=20,
    min_samples_leaf=3,
    min_samples_split=2,
    max_features='sqrt',
    n_estimators=50,
)

model_rf_base = Pipeline(steps=[
    ('prep', preprocess),
    ('model', rf_base),
])

model_rf_base.fit(X_train, y_train)
pred_rf_base = model_rf_base.predict(X_test)

mae_rf_base = mean_absolute_error(y_test, pred_rf_base)
rmse_rf_base = root_mean_squared_error(y_test, pred_rf_base)
r2_rf_base = r2_score(y_test, pred_rf_base)

print(f"Random Forest — MAE = {mae_rf_base:.2f}, RMSE = {rmse_rf_base:.2f}, R² = {r2_rf_base:.3f}")

Random Forest — MAE = 16.25, RMSE = 30.21, R² = 0.933


## Entraînement de modèle (2) : Random Forest optimisé

In [7]:
model_rf_grid = Pipeline(steps=[('prep', preprocess), ('model', rf_base)])

max_depth_list = [10, 20, 30]
min_samples_leaf_list = [5, 10]
min_samples_split_list = [10, 20]
max_features_list = ['sqrt', 'log2']
n_estimators_list = [10, 15, 20]

results = []

for max_depth_grid in max_depth_list:
    for min_samples_leaf_grid in min_samples_leaf_list:
        for min_samples_split_grid in min_samples_split_list:
            for max_features_grid in max_features_list:
                for n_estimators_grid in n_estimators_list:

                    print(
                        f"\nEntraînement du modèle avec max_depth = {max_depth_grid}, "
                        f"min_samples_leaf = {min_samples_leaf_grid}, "
                        f"min_samples_split = {min_samples_split_grid}, "
                        f"max_features = {max_features_grid}, "
                        f"n_estimators = {n_estimators_grid}"
                    )

                    model_rf_grid.set_params(
                        model__max_depth=max_depth_grid,
                        model__min_samples_leaf=min_samples_leaf_grid,
                        model__min_samples_split=min_samples_split_grid,
                        model__max_features=max_features_grid,
                        model__n_estimators=n_estimators_grid,
                    )

                    start_time = time.time()
                    model_rf_grid.fit(X_train, y_train)
                    train_time = time.time() - start_time

                    pred_rf_grid = model_rf_grid.predict(X_test)

                    mae_rf_grid = mean_absolute_error(y_test, pred_rf_grid)
                    rmse_rf_grid = root_mean_squared_error(y_test, pred_rf_grid)
                    r2_rf_grid = r2_score(y_test, pred_rf_grid)

                    print(
                        f"-> Entraînement effectué en {train_time:.2f}s | "
                        f"MAE = {mae_rf_grid:.2f}, RMSE = {rmse_rf_grid:.2f}, R² = {r2_rf_grid:.3f}"
                    )

                    results.append({
                        'max_depth': max_depth_grid,
                        'min_samples_leaf': min_samples_leaf_grid,
                        'min_samples_split': min_samples_split_grid,
                        'max_features': max_features_grid,
                        'n_estimators': n_estimators_grid,
                        'MAE': mae_rf_grid,
                        'RMSE': rmse_rf_grid,
                        'R2': r2_rf_grid,
                        'train_time_sec': train_time,
                    })

df_results = pd.DataFrame(results)

df_results_sorted = df_results.sort_values(by='R2', ascending=False)


Entraînement du modèle avec max_depth = 10, min_samples_leaf = 5, min_samples_split = 10, max_features = sqrt, n_estimators = 10
-> Entraînement effectué en 0.88s | MAE = 24.94, RMSE = 46.31, R² = 0.841

Entraînement du modèle avec max_depth = 10, min_samples_leaf = 5, min_samples_split = 10, max_features = sqrt, n_estimators = 15
-> Entraînement effectué en 1.30s | MAE = 24.17, RMSE = 44.32, R² = 0.855

Entraînement du modèle avec max_depth = 10, min_samples_leaf = 5, min_samples_split = 10, max_features = sqrt, n_estimators = 20
-> Entraînement effectué en 1.42s | MAE = 23.36, RMSE = 42.82, R² = 0.864

Entraînement du modèle avec max_depth = 10, min_samples_leaf = 5, min_samples_split = 10, max_features = log2, n_estimators = 10
-> Entraînement effectué en 0.80s | MAE = 26.00, RMSE = 46.69, R² = 0.839

Entraînement du modèle avec max_depth = 10, min_samples_leaf = 5, min_samples_split = 10, max_features = log2, n_estimators = 15
-> Entraînement effectué en 1.11s | MAE = 27.58, RMSE 

In [8]:
df_results_sorted.to_csv('../data/processed/df_random_forest_tuning.csv', index=False, encoding='utf-8')

In [9]:
#df_results_sorted = pd.read_csv('../data/processed/df_random_forest_tuning.csv')

df_results_sorted.loc[(df_results_sorted['R2'] >= 0.90) & (df_results_sorted['train_time_sec'] <= 10)]

,max_depth,min_samples_leaf,min_samples_split,max_features,n_estimators,MAE,RMSE,R2,train_time_sec
54,30,5,20,sqrt,10,16.113034,30.359449,0.931844,7.999661
66,30,10,20,sqrt,10,16.267783,31.081456,0.928564,6.981739
60,30,10,10,sqrt,10,16.267783,31.081456,0.928564,6.567802
26,20,5,10,sqrt,20,16.863400,31.363154,0.927263,7.960236
25,20,5,10,sqrt,15,17.107146,31.705452,0.925666,6.547053
44,20,10,20,sqrt,20,17.247317,32.377142,0.922484,6.016491
38,20,10,10,sqrt,20,17.247317,32.377142,0.922484,6.258246
24,20,5,10,sqrt,10,17.548725,32.416164,0.922297,4.703582
43,20,10,20,sqrt,15,17.368457,32.491886,0.921933,4.675862
37,20,10,10,sqrt,15,17.368457,32.491886,0.921933,4.773582


In [10]:
rf_tuned = RandomForestRegressor(
    n_jobs=-1,
    random_state=42,
    verbose=0,
    max_depth=30,
    min_samples_leaf=5,
    min_samples_split=20,
    max_features='sqrt',
    n_estimators=10,
)

model_rf_tuned = Pipeline(steps=[
    ('prep', preprocess),
    ('model', rf_tuned),
])

model_rf_tuned.fit(X_train, y_train)
pred_rf_tuned_test = model_rf_tuned.predict(X_test)

mae_rf_tuned = mean_absolute_error(y_test, pred_rf_tuned_test)
rmse_rf_tuned = root_mean_squared_error(y_test, pred_rf_tuned_test)
r2_rf_tuned_test = r2_score(y_test, pred_rf_tuned_test)

pred_rf_tuned_train = model_rf_tuned.predict(X_train)
r2_rf_tuned_train = r2_score(y_train, pred_rf_tuned_train)

print(
    f"Random Forest — MAE = {mae_rf_tuned:.2f}, RMSE = {rmse_rf_tuned:.2f}, "
    f"R²-test = {r2_rf_tuned_test:.3f}, R²-train = {r2_rf_tuned_train:.3f}"
)

models_dir = Path('../models')
models_dir.mkdir(exist_ok=True)

joblib.dump(model_rf_tuned, models_dir / 'model_rf.joblib', compress=('gzip', 3))

Random Forest — MAE = 16.11, RMSE = 30.36, R²-test = 0.932, R²-train = 0.954


['../models/model_rf.joblib']

## Conclusion

Observations sur le dataset de mai 2024 à juin 2025 :

J’ai commencé par un premier Random Forest simple, puis j’ai testé plusieurs paramètres pour trouver un bon compromis entre performance et temps d’entraînement (moins de 10 secondes).

La meilleure version obtient une **MAE de 16.46, une RMSE de 30.88 et un R²-test de 0.929**, ce qui représente une légère amélioration par rapport au modèle de départ.

Comparées aux autres approches, les différences sont plus nettes : les modèles basés sur les lags atteignaient un R² maximal de 0.782, et les modèles linéaires montaient jusqu’à 0.895. Le Random Forest parvient bien mieux à capturer la variance et la structure complexe des données.